In [12]:
%pip install pandas openai google-generativeai

  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/86/41/585a168330ff063014880a80d744219dbf1dd7a1c706e75ab3425a987384/pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/59/fd/ae2da789cd923dd033c99b8d544071a827c92046b150db01cfa5cea5b3fd/openai-2.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for google-generativeai from https://files.pythonhosted.org/packages/6e/40/c42ff9ded9f09ec9392879a8e6538a00b2dc185e834a3392917626255419/google_generativeai-0.8.5-py3-none-any.whl.metadata
  Obtaining dependency information for numpy>=1.26.0 from https://files.pythonhosted.org/packages/2d/57/8aeaf160312f7f489dea47ab61e430b5cb051f59a98ae68b7133ce8fa06a/numpy-2.3.5-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/60.9 kB ? eta -:--:--
     ---------------------------------------- 60.9/60.9 kB 1.6 MB/s eta 0:00:00
  Obtainin

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import random
import re
import time
import pandas as pd


c:\Users\krish\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def load_and_sample_data(filepath, sample_size=200, random_state=42):
    """
    Loads the Yelp dataset and samples a subset for evaluation.
    """
    try:
        df = pd.read_csv(filepath)
        # Ensure we have the necessary columns
        if 'text' not in df.columns or 'stars' not in df.columns:
            raise ValueError("Dataset must contain 'text' and 'stars' columns.")
        
        # Sample data
        if len(df) > sample_size:
            df_sampled = df.sample(n=sample_size, random_state=random_state)
        else:
            df_sampled = df
            
        return df_sampled
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


In [15]:
def get_system_prompt():
    return "You are a helpful assistant that analyzes Yelp reviews."

def strategy_1_zeroshot(review_text):
    """
    Strategy 1: Zero-Shot / Baseline
    Directly asks for the rating and explanation in JSON format.
    """
    prompt = f"""
    Analyze the following Yelp review and determine the star rating (1-5).
    Provide the output in strict JSON format with keys "predicted_stars" (integer) and "explanation" (string).
    
    Review: "{review_text}"
    
    JSON Output:
    """
    return prompt

def strategy_2_fewshot(review_text):
    """
    Strategy 2: Few-Shot
    Provides examples of reviews and their ratings to guide the model.
    """
    prompt = f"""
    Determine the star rating (1-5) for the Yelp review below.
    Return JSON format: {{"predicted_stars": <int>, "explanation": "<string>"}}

    Examples:
    Review: "The food was okay, but the service was terrible. I waited 30 minutes for water."
    Output: {{"predicted_stars": 2, "explanation": "Food was average but service was very poor."}}

    Review: "Absolutely loved it! The steak was cooked to perfection and the staff was friendly."
    Output: {{"predicted_stars": 5, "explanation": "Positive comments on both food and service."}}

    Review: "It was decent. Nothing special, but good for a quick bite."
    Output: {{"predicted_stars": 3, "explanation": "Average experience, met expectations but didn't exceed them."}}

    Review: "{review_text}"
    Output:
    """
    return prompt

def strategy_3_cot(review_text):
    """
    Strategy 3: Chain-of-Thought
    Asks the model to reason step-by-step before assigning a rating.
    """
    prompt = f"""
    Analyze the following Yelp review to determine the star rating (1-5).
    
    Review: "{review_text}"
    
    Instructions:
    1. Identify the key sentiment-bearing phrases in the review.
    2. Analyze the sentiment towards specific aspects (food, service, ambiance, value).
    3. Weigh the positive and negative points.
    4. Assign a star rating based on the overall sentiment.
    
    Finally, output the result in strict JSON format:
    {{
        "predicted_stars": <int>,
        "explanation": "<brief reasoning>"
    }}
    """
    return prompt


In [ ]:
class MockLLM:
    """
    A mock LLM for testing purposes when no API key is available.
    Uses simple keyword matching to guess sentiment.
    """
    def generate(self, prompt):
        # Extract the review text from the prompt (simplified)
        review_match = re.search(r'Review: "(.*?)"', prompt, re.DOTALL)
        if review_match:
            text = review_match.group(1).lower()
        else:
            text = ""
            
        # Simple heuristic
        positive_words = ['great', 'good', 'love', 'excellent', 'amazing', 'best', 'awesome', 'delicious']
        negative_words = ['bad', 'terrible', 'worst', 'hate', 'awful', 'disgusting', 'poor', 'slow']
        
        score = 3
        for word in positive_words:
            if word in text:
                score += 1
        for word in negative_words:
            if word in text:
                score -= 1
                
        score = max(1, min(5, score))
        
        # Simulate JSON output
        response = {
            "predicted_stars": score,
            "explanation": "This is a MOCK explanation based on keyword heuristics."
        }
        return json.dumps(response)


class GeminiLLM:
    """
    Wrapper for Google Gemini API.
    """
    def __init__(self, api_key, model="gemini-2.5-flash"):
        self.api_key = api_key
        self.model_name = model
        try:
            import google.generativeai as genai
            genai.configure(api_key=self.api_key)
            self.model = genai.GenerativeModel(self.model_name)
            self.genai = genai
        except ImportError:
            print("Google Generative AI library not installed. Please install with `pip install google-generativeai`.")
            self.model = None

    def generate(self, prompt):
        if not self.model:
            return json.dumps({"predicted_stars": 0, "explanation": "Gemini lib missing"})
            
        max_retries = 5
        base_delay = 2
        
        for attempt in range(max_retries):
            try:
                response = self.model.generate_content(prompt)
                return response.text
            except Exception as e:
                error_str = str(e)
                if "429" in error_str or "quota" in error_str.lower():
                    delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                    print(f"Rate limit hit. Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(delay)
                else:
                    print(f"Gemini API Error: {e}")
                    return json.dumps({"predicted_stars": 0, "explanation": f"API Error: {e}"})
        
        return json.dumps({"predicted_stars": 0, "explanation": "Rate limit exceeded after retries"})


In [ ]:
def evaluate_strategy(strategy_func, df, llm):
    """
    Runs a specific prompting strategy on the dataframe.
    """
    results = []
    valid_json_count = 0
    correct_count = 0
    
    print(f"Evaluating {strategy_func.__name__}...")
    
    for _, row in df.iterrows():
        review_text = row['text']
        actual_stars = row['stars']
        
        prompt = strategy_func(review_text)
        response_text = llm.generate(prompt)
        
        # Parse JSON
        try:
            # clean up markdown code blocks if present
            cleaned_response = response_text.replace("```json", "").replace("```", "").strip()
            data = json.loads(cleaned_response)
            predicted_stars = data.get("predicted_stars")
            explanation = data.get("explanation")
            valid_json_count += 1
            
            if predicted_stars == actual_stars:
                correct_count += 1
                
        except json.JSONDecodeError:
            predicted_stars = None
            explanation = "JSON Decode Error"
            
        results.append({
            "actual": actual_stars,
            "predicted": predicted_stars,
            "explanation": explanation,
            "valid_json": predicted_stars is not None
        })
        
        # Add a small delay between requests to be safe
        time.sleep(2)
        
    accuracy = correct_count / len(df) if len(df) > 0 else 0
    json_validity = valid_json_count / len(df) if len(df) > 0 else 0
    
    return {
        "accuracy": accuracy,
        "json_validity": json_validity,
        "details": results
    }


In [ ]:
df = load_and_sample_data("yelp.csv", sample_size=10)

if df is not None:
    # 2. Setup LLM
    gemini_key = os.getenv("GEMINI_API_KEY")

    if gemini_key:
        print("Using Gemini API.")
        llm = GeminiLLM(gemini_key)
    else:
        print("No API key found. Using Mock LLM (Keyword Heuristic).")
        print("To use real LLM, set GEMINI_API_KEY or OPENAI_API_KEY environment variable.")
        llm = MockLLM()

    # 3. Define Strategies
    strategies = [
        strategy_1_zeroshot,
        strategy_2_fewshot,
        strategy_3_cot
    ]

    # 4. Run Evaluation
    summary_table = []

    for strategy in strategies:
        metrics = evaluate_strategy(strategy, df, llm)
        summary_table.append({
            "Strategy": strategy.__name__,
            "Accuracy": f"{metrics['accuracy']:.2%}",
            "JSON Validity": f"{metrics['json_validity']:.2%}"
        })

        print(f"\n--- Example Output for {strategy.__name__} ---")
        if metrics['details']:
            print(metrics['details'][0])
        print("------------------------------------------------\n")

    # 5. Comparison Table
    print("\n=== Evaluation Results ===")
    df_summary = pd.DataFrame(summary_table)
    print(df_summary.to_string(index=False))


Loading data...
Using Gemini API.
Evaluating strategy_1_zeroshot...

--- Example Output for strategy_1_zeroshot ---
{'actual': 4, 'predicted': 4, 'explanation': "The review is overwhelmingly positive. While the reviewer notes the place was 'dead' late at night, they provide an understanding explanation (location, time) and do not fault the establishment. Instead, they highlight numerous strengths: 'well made pub grub,' 'friendly service,' 'quality cocktails,' and an atmosphere that 'certainly works for a sports bar.' They also mention it's a 'great spot for happy hour' and busy through 10 pm, with a 'great patio for day-drinking.' These consistent positives, with no actual complaints, strongly indicate a very good experience, meriting a 4-star rating.", 'valid_json': True}
------------------------------------------------

Evaluating strategy_2_fewshot...

--- Example Output for strategy_2_fewshot ---
{'actual': 4, 'predicted': 5, 'explanation': "The review is overwhelmingly positive, p